In [6]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime
import pytz
import os

try:
    # MT5に接続
    if not mt5.initialize():
        print("initialize() failed, error code =", mt5.last_error())
        quit()

    # 設定
    symbol = "USDJPY"
    timeframe = mt5.TIMEFRAME_M1
    timezone = pytz.timezone("Etc/UTC")
    utc_from = datetime(2020, 1, 1, tzinfo=timezone)
    utc_to = datetime(2020, 12, 31, hour=13, tzinfo=timezone)

    # ポイント単位（1ポイント = 0.001など）取得
    symbol_info = mt5.symbol_info(symbol)
    if symbol_info is None:
        raise Exception(f"Failed to get symbol info for {symbol}")
    point = symbol_info.point  # USDJPY は通常 0.001

    # データ取得
    rates = mt5.copy_rates_range(symbol, timeframe, utc_from, utc_to)
    if rates is None:
        raise Exception("No data received, error code = {}".format(mt5.last_error()))

    # DataFrameに変換
    df = pd.DataFrame(rates, columns=[
        'time', 'open', 'high', 'low', 'close',
        'tick_volume', 'spread', 'real_volume'
    ])

    # タイムスタンプ変換
    df['time'] = pd.to_datetime(df['time'], unit='s')

    # ✅ スプレッドを「価格単位」に変換（上書き）
    df['spread'] = df['spread'] * point

    # 保存フォルダ作成
    csv_folder = 'csv'
    if not os.path.exists(csv_folder):
        os.makedirs(csv_folder)

    # 保存ファイル名
    from_date_str = utc_from.strftime("%Y%m%d")
    to_date_str = utc_to.strftime("%Y%m%d")
    csv_file = os.path.join(csv_folder, f'{symbol}_{timeframe}_{from_date_str}_to_{to_date_str}.csv')
    df.to_csv(csv_file, index=False)

    print(f"{csv_file} has been saved.")

except Exception as e:
    print("An error occurred:", str(e))

finally:
    mt5.shutdown()


csv\USDJPY_1_20200101_to_20201231.csv has been saved.


In [7]:
!python --version

Python 3.9.7
